# Competencia — Aprendizaje de Máquina 2026-10
## Parte 2: Clasificación de Textos Históricos con Deep Learning

Este notebook implementa la Parte 2 del proyecto de clasificación de textos históricos
en español/latín según su **década de origen** (39 clases, de 1500 a 1880). El objetivo
es superar el Private Score de referencia de la Parte 1 (0.29144) usando arquitecturas
de deep learning y transferencia de aprendizaje.

El enfoque adopta una estrategia en dos niveles diseñada para entrenamiento en CPU:

- **Nivel 1 — MLP sobre TF-IDF:** red neuronal densa que reemplaza el LinearSVC de la
  Parte 1 sobre la misma representación TF-IDF. Cumple el requisito de arquitectura de
  deep learning y establece un punto de comparación interno directo con la Parte 1.

- **Nivel 2 — Transfer Learning con MiniLM:** modelo principal. Se usa
  `paraphrase-multilingual-MiniLM-L12-v2`, un transformer multilingüe destilado de
  BERT entrenado en más de 50 idiomas (incluyendo español y latín). Genera embeddings
  densos de 384 dimensiones que capturan semántica contextual imposible de representar
  con TF-IDF. Sobre estos embeddings se entrena una cabeza clasificadora MLP con las
  41 features lingüísticas de la Parte 1 concatenadas. Esta arquitectura satisface los
  requisitos de deep learning, transferencia de aprendizaje y es ejecutable en CPU en
  2-4 horas sobre el dataset completo.

**Métrica objetivo:** Accuracy (leaderboard Kaggle) y F1-macro (optimización interna).

## 1. 📦 Instalación de Dependencias

Hacer el .venv con la version 3.11 de python


In [1]:
import subprocess
import sys
from pathlib import Path

def run(cmd: list) -> bool:
    print(f"⏳ {' '.join(cmd)}")
    result = subprocess.run(cmd)
    return result.returncode == 0

print("=" * 70)
print("📦 INSTALADOR DE DEPENDENCIAS")
print("=" * 70)
print(f"Python activo : {sys.executable}")
print(f"Versión       : {sys.version.split()[0]}")
print(f"Proyecto      : {Path.cwd()}")
print("=" * 70)

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

BASE_PACKAGES = [
    "numpy", "pandas", "matplotlib",
    "seaborn", "scikit-learn", "tqdm",
]

print("\n🧩 Instalando paquetes base...\n")
for pkg in BASE_PACKAGES:
    ok = run([sys.executable, "-m", "pip", "install", pkg])
    if not ok:
        print(f"❌ Falló: {pkg}")

# Todo ML en un solo pip install con versiones fijas y compatibles
print("\n🧠 Instalando paquetes de ML...\n")
ok = run([sys.executable, "-m", "pip", "install",
    "torch",
    "sentencepiece",
    "tokenizers==0.19.1",
    "huggingface-hub==0.23.4",
    "transformers==4.40.0",
    "sentence-transformers==3.0.0",
    "datasets==2.19.0",
    "accelerate==0.29.0",
    "--no-deps",   # NO instala dependencias que pisen las versiones fijadas
])
# Instalar deps faltantes por --no-deps
run([sys.executable, "-m", "pip", "install",
    "torch", "numpy", "scipy", "tqdm", "requests", "packaging",
])

if not ok:
    print("❌ Falló ML")

print("\n✅ Listo")
run([sys.executable, "-m", "pip", "show", "huggingface-hub"])

📦 INSTALADOR DE DEPENDENCIAS
Python activo : /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML/.venv312/bin/python
Versión       : 3.9.20
Proyecto      : /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML
⏳ /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML/.venv312/bin/python -m pip install --upgrade pip setuptools wheel

🧩 Instalando paquetes base...

⏳ /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML/.venv312/bin/python -m pip install numpy
⏳ /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML/.venv312/bin/python -m pip install pandas
⏳ /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML/.venv312/bin/python -m pip install matplotlib
⏳ /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML/.venv312/bin/python -m pip install seaborn
⏳ /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML/.venv312/bin/python -m pip install scikit-learn
⏳ /Users/juancamilocaldas/Desktop/ProyectoML_2/Proyecto_2-ML/.venv312/bin/python -m pip instal

True

In [2]:
import pandas as pd
import numpy as np
import re, os, math, warnings, random
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from scipy.sparse import hstack, csr_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt

# ── Reproducibilidad ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Constantes del experimento ──
BASELINE_P1_LOCAL   = 0.2737   # F1-macro del mejor modelo Parte 1 (80/20 local)
BASELINE_P1_KAGGLE  = 0.29144  # Private Score Kaggle Parte 1 — score a superar
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

os.makedirs('./submissions', exist_ok=True)
os.makedirs('./model', exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device:  {DEVICE}')
print('✅ Listo')

W0515 17:21:44.015251 63692 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


PyTorch: 2.8.0
Device:  mps
✅ Listo


## 1. Carga de Datos

El dataset es idéntico al de la Parte 1:
- `train.csv`: 31.403 textos etiquetados con su década de origen
- `eval.csv`: 3.490 textos sin etiqueta sobre los cuales se generan las predicciones finales

La variable objetivo `decade` representa los tres primeros dígitos del año (ej: `164` para
la década de 1640), dando un total de **39 clases** desde 150 hasta 188.

In [3]:
df_train = pd.read_csv('./data/train.csv')
df_eval  = pd.read_csv('./data/eval.csv')

print(f'Train: {df_train.shape} | Eval: {df_eval.shape}')
print(f'Clases únicas: {df_train["decade"].nunique()}')
print(f'Distribución de clases (min/max textos): '
      f'{df_train["decade"].value_counts().min()} / '
      f'{df_train["decade"].value_counts().max()}')

Train: (31403, 2) | Eval: (3490, 2)
Clases únicas: 39
Distribución de clases (min/max textos): 754 / 848


El dataset contiene 31.403 textos de entrenamiento distribuidos en 39 clases con una
distribución casi uniforme — entre 754 y 848 textos por década. Esta uniformidad es
favorable para el entrenamiento: no hay sesgo por desbalance de clases y no se requiere
class weighting ni oversampling.

## 2. Preprocesamiento — Normalización OCR

Se reutiliza la función `normalize_ocr` de la Parte 1, que corrige artefactos tipográficos
y de digitalización sin tocar la ortografía arcaica del texto. El tokenizador de MiniLM
opera a nivel de subpalabra (WordPiece), por lo que formas como `hazer`, `dize` o `vna`
se tokenizarán en sus propios subwords — preservar esta señal es crítico para que el
modelo aprenda la evolución temporal del español.

Se eliminan además los 51 duplicados detectados en la Parte 1, quedando **31.352 textos**
para entrenamiento.

In [4]:
# ── Mapa de sustituciones OCR (idéntico a Parte 1) ──
CHAR_MAP = [
    ('\ufb01','fi'),('\ufb02','fl'),('\ufb00','ff'),('\ufb03','ffi'),('\ufb04','ffl'),
    ('\xe6','ae'),('\u0153','oe'),
    ('-\n',''),('- \n',''),('\xad',''),
    ('\xbb',' '),('\xab',' '),
    ('\u2018',"'"),("\u2019","'"),("\u201c",'"'),("\u201d",'"'),
    ('\xa3',' '),('\xa7',' '),('\xb6',' '),
    ('\u2020',' '),('\u2021',' '),('\u2022',' '),
    ('\u2014',' '),('\u2013',' '),
]

def normalize_ocr(text):
    text = str(text)
    for src, tgt in CHAR_MAP:
        text = text.replace(src, tgt)
    text = text.replace('\n',' ').replace('\r',' ').replace('\t',' ')
    return re.sub(r'  +', ' ', text).strip()

# ── Aplicar normalización y eliminar duplicados ──
data = df_train.drop_duplicates(subset='text').reset_index(drop=True).copy()
data['text_clean']    = data['text'].apply(normalize_ocr)
df_eval['text_clean'] = df_eval['text'].apply(normalize_ocr)

print(f'Train sin duplicados: {len(data)}')
print(f'Duplicados eliminados: {len(df_train) - len(data)}')

Train sin duplicados: 31352
Duplicados eliminados: 51


Se confirmaron los 51 duplicados detectados en la Parte 1. El dataset queda en 31.352
textos limpios. La normalización OCR preserva la ortografía arcaica intacta — las
sustituciones solo corrigen artefactos de digitalización (ligaduras, guiones de corte
de línea, símbolos tipográficos) que no aportan señal temporal.

## 3. Extracción de Features Lingüísticas Históricas

Se extienden las **41 features numéricas** de la Parte 1 con **39 features adicionales**
agrupadas en tres categorías, para un total de **80 features por texto**.

Las features se concatenarán más adelante a los embeddings de MiniLM antes de la capa
clasificadora, aportando señal explícita que el transformer no captura directamente
desde el texto crudo. La motivación es triple:

- **Ortografía arcaica ampliada (18):** patrones regex que discriminan décadas con alta
  precisión — *cedilla* y *hazer* señalan el XVI, *ahora* y *mismo* el XVIII–XIX.
- **Sintaxis superficial (8):** longitud y varianza de oraciones, conectores ilustrados
  y densidad de gerundios aproximan la complejidad sintáctica sin parseo completo.
- **Léxico de época (13):** diccionarios de arcaísmos (XVI–XVII), vocabulario ilustrado
  (XVIII) y léxico realista (XIX) extraídos del CORDE, con ratios relativos entre épocas.

Las 41 features base se mantienen idénticas a la Parte 1 para preservar comparabilidad.

In [5]:
### 3.1 Features Lingüísticas Extendidas
import re, math, numpy as np
from collections import Counter

# ── A) Ortografía arcaica ampliada ────────────────────────────────────────────
RE_HAZER     = re.compile(r'\bha[zs]er\b|\bdi[zs]e\b|\bdi[zs]en\b|\bdixo\b|\btraxo\b', re.I)
RE_VOS       = re.compile(r'\bvos\b|\bvuestra\s+merced\b|\bv\.m\b', re.I)
RE_AUIA      = re.compile(r'\bau[ií]a\b|\bhau[ií]a\b|\baueys\b|\bauéis\b', re.I)
RE_CEDILLA   = re.compile(r'[çÇ]|\bç[aouáóú]', re.I)
RE_EL_FEM    = re.compile(r'\bel\s+(?:alma|agua|arte|ave|ánima|anima)\b', re.I)
RE_APOCC     = re.compile(r'\bgrand\b|\bsant\b|\bgran\b(?=\s+[bcdfghjklmnpqrstvwxyz])', re.I)
RE_PH        = re.compile(r'\bph[aeiouáéíóú]', re.I)
RE_MENTE     = re.compile(r'\w+mente\b', re.I)
RE_NON       = re.compile(r'\bnon\b', re.I)
RE_AGORA     = re.compile(r'\bagora\b', re.I)
RE_AHORA     = re.compile(r'\bahora\b', re.I)
RE_MESMO     = re.compile(r'\bme[sz]mo\b|\bme[sz]ma\b', re.I)
RE_MISMO     = re.compile(r'\bmismo\b|\bmisma\b', re.I)
RE_AVER      = re.compile(r'\baver\b|\bovo\b|\boviste\b', re.I)
RE_FORA      = re.compile(r'\bfora\b|\bseyer\b|\bseer\b', re.I)
RE_DO        = re.compile(r'\bdo\b(?=\s+\w)', re.I)
RE_IA_END    = re.compile(r'\b\w{4,}[íi]a\b', re.I)
RE_LATIN_RAW = re.compile(r'\b(status|corpus|modus|locus|nexus|census|apparatus)\b', re.I)

# ── B) Sintaxis superficial ───────────────────────────────────────────────────
RE_SUBORD    = re.compile(r'\b(que|cual|quien|donde|cuando|como|si|aunque|sino|porque|pues)\b', re.I)
RE_DISC      = re.compile(r'\b(asimismo|empero|por\s+ende|por\s+tanto|no\s+obstante|'
                            r'sin\s+embargo|en\s+efecto|a\s+saber|es\s+decir|esto\s+es)\b', re.I)
RE_REL       = re.compile(r'\b(el|la|los|las)\s+que\b', re.I)
RE_INV       = re.compile(r'\b(dijo|respondió|exclamó|preguntó|añadió)\s+\w+\b', re.I)
RE_GERUND    = re.compile(r'\b\w+(ando|iendo)\b', re.I)
RE_INF_PREP  = re.compile(r'\b(al|sin|por|para|de|a)\s+\w+(ar|er|ir)\b', re.I)

# ── C) Léxico de época ────────────────────────────────────────────────────────
ARCAISMOS_XVI_XVII = {
    'ansí','assí','assi','ansina','muncho','agora','aora',
    'mesmo','mesma','mesmos','mesmas','ovo','ove','oviste','ovimos',
    'desque','desquel','ca','do','dó','maguer','maguera',
    'ende','dende','cabe','so','fasta','fata',
    'primeramente','quisiera','pluguiera','facerlo','fazerlo',
}
LEXICO_XVIII = {
    'razón','racional','racionalidad','ilustración','ilustrado','filosofía',
    'naturaleza','natural','física','experiencia','experimento','observación',
    'utilidad','útil','progreso','nación','patria','ciudadano',
    'reforma','reformar','reformas','educación','instrucción',
    'opinión','público','sociedad','comercio','industria','agricultura',
    'felicidad','superstición','ignorancia','credulidad',
}
LEXICO_XIX = {
    'burgués','burguesía','clase','dinero','capital','negocio',
    'ferrocarril','telégrafo','vapor','periódico','prensa','redacción',
    'republicano','liberal','conservador','fábrica','obrero','trabajo',
    'señorita','señora','caballero','salón','visita','tertulia',
    'novela','romance','sentimental','clínica','médico','enfermedad',
    'realismo','naturalismo',
}

def count_lexicon(wl, lexicon):
    return sum(1 for w in wl if w in lexicon)

def extract_features_extended(text):
    words   = text.split()
    n       = max(len(words), 1)
    nc      = max(len(text), 1)
    lengths = [len(w) for w in words]
    counts  = Counter(text)
    total   = len(text) or 1
    entropy = -sum((c/total)*math.log2(c/total) for c in counts.values()) if text else 0
    vowels  = sum(1 for c in text.lower() if c in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    cons    = sum(1 for c in text.lower() if c.isalpha() and c not in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    sents   = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    ns      = max(len(sents), 1)
    sent_lengths = [len(s.split()) for s in sents]
    wl      = [w.lower().strip('.,;:!?()[]{}"\'-') for w in words]

    RE_LONG_S    = re.compile(r'[bcdfghjklmnpqrstvwxyz]f[aeiouáéíóú]', re.I)
    RE_V_AS_U    = re.compile(r'\bvn[aeiouáéíóú]|\bvn\b|\bvm\b', re.I)
    RE_ROMAN     = re.compile(r'\b(M{1,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})|CM|CD|XC|XL|IX|IV|D?C{2,3}|L?X{2,3}|V?I{2,4})\b')
    RE_LATIN_END = re.compile(r'\b\w{3,}(orum|ibus|atis|endi|antis|entis)\b', re.I)
    RE_LATIN_NOM = re.compile(r'\b\w{3,}(um|us|ae)\b', re.I)
    RE_QU_ARC    = re.compile(r'\bqu[aou]\w', re.I)
    RE_SS        = re.compile(r'ss', re.I)
    RE_FF        = re.compile(r'ff', re.I)
    RE_CION      = re.compile(r'\b\w{3,}cion\b', re.I)
    RE_TION      = re.compile(r'\b\w{3,}tion\b', re.I)
    RE_SION      = re.compile(r'\b\w{3,}sion\b', re.I)
    RE_ABBREV    = re.compile(r'\b[A-Za-z]{1,4}\.')
    RE_MCASE     = re.compile(r'\b[A-Z][a-z]{1,}[A-Z]\w*\b')
    RE_RDIAC     = re.compile(r'[àâãäāăąèêëēĕěîïīĭôõōŏùûüūŭ]', re.I)
    RE_SEMI      = re.compile(r';')
    RE_COLON     = re.compile(r':')
    RE_PAREN     = re.compile(r'[()]')

    arc = count_lexicon(wl, ARCAISMOS_XVI_XVII)
    xvi = count_lexicon(wl, LEXICO_XVIII)
    xix = count_lexicon(wl, LEXICO_XIX)

    return {
        # ── 41 base ───────────────────────────────────────────────────────────
        'wl_mean':        np.mean(lengths) if lengths else 0,
        'wl_std':         np.std(lengths)  if lengths else 0,
        'wl_p75':         np.percentile(lengths, 75) if lengths else 0,
        'wl_p90':         np.percentile(lengths, 90) if lengths else 0,
        'ratio_long':     sum(1 for l in lengths if l > 10) / n,
        'ratio_short':    sum(1 for l in lengths if l <= 2) / n,
        'ratio_med':      sum(1 for l in lengths if 3 <= l <= 6) / n,
        'char_entropy':   entropy,
        'cv_ratio':       cons / vowels if vowels > 0 else 0,
        'word_char_ratio':n / nc,
        'comma_rate':     text.count(',') / n,
        'period_rate':    text.count('.') / n,
        'semicolon_rate': len(RE_SEMI.findall(text))  / n,
        'colon_rate':     len(RE_COLON.findall(text)) / n,
        'paren_rate':     len(RE_PAREN.findall(text)) / n,
        'excl_rate':      text.count('!') / n,
        'quest_rate':     text.count('?') / n,
        'total_punct':    sum(1 for c in text if c in '.,;:!?()[]{}') / n,
        'long_s_rate':    len(RE_LONG_S.findall(text))    / n,
        'v_as_u_rate':    len(RE_V_AS_U.findall(text))    / n,
        'latin_case':     len(RE_LATIN_END.findall(text)) / n,
        'latin_nom':      len(RE_LATIN_NOM.findall(text)) / n,
        'qu_archaic':     len(RE_QU_ARC.findall(text))    / n,
        'ss_rate':        len(RE_SS.findall(text))         / n,
        'ff_rate':        len(RE_FF.findall(text))         / n,
        'cion_rate':      len(RE_CION.findall(text))       / n,
        'tion_rate':      len(RE_TION.findall(text))       / n,
        'sion_rate':      len(RE_SION.findall(text))       / n,
        'abbrev_rate':    len(RE_ABBREV.findall(text))     / n,
        'roman_rate':     len(RE_ROMAN.findall(text))      / n,
        'rare_diac':      len(RE_RDIAC.findall(text))      / nc,
        'mixed_case':     len(RE_MCASE.findall(text))      / n,
        'ttr':            len(set(w.lower() for w in words)) / n,
        'upper_ratio':    sum(1 for c in text if c.isupper()) / nc,
        'digit_ratio':    sum(1 for c in text if c.isdigit()) / nc,
        'alpha_ratio':    sum(1 for c in text if c.isalpha()) / nc,
        'all_caps':       sum(1 for w in words if w.isupper() and len(w)>1) / n,
        'pure_digit':     sum(1 for w in words if w.isdigit()) / n,
        'n_words':        float(n),
        'avg_sent_len':   n / ns,
        'n_sentences':    float(ns),
        # ── 18 ortografía arcaica ─────────────────────────────────────────────
        'hazer_rate':     len(RE_HAZER.findall(text))     / n,
        'vos_rate':       len(RE_VOS.findall(text))        / n,
        'auia_rate':      len(RE_AUIA.findall(text))       / n,
        'cedilla_rate':   len(RE_CEDILLA.findall(text))    / nc,
        'el_fem_rate':    len(RE_EL_FEM.findall(text))     / n,
        'apocc_rate':     len(RE_APOCC.findall(text))      / n,
        'ph_rate':        len(RE_PH.findall(text))         / n,
        'non_rate':       len(RE_NON.findall(text))        / n,
        'agora_rate':     len(RE_AGORA.findall(text))      / n,
        'mesmo_rate':     len(RE_MESMO.findall(text))      / n,
        'aver_rate':      len(RE_AVER.findall(text))       / n,
        'fora_rate':      len(RE_FORA.findall(text))       / n,
        'do_arc_rate':    len(RE_DO.findall(text))         / n,
        'latin_raw_rate': len(RE_LATIN_RAW.findall(text))  / n,
        'mente_rate':     len(RE_MENTE.findall(text))      / n,
        'ahora_rate':     len(RE_AHORA.findall(text))      / n,
        'mismo_rate':     len(RE_MISMO.findall(text))      / n,
        'ia_end_rate':    len(RE_IA_END.findall(text))     / n,
        # ── 8 sintaxis superficial ────────────────────────────────────────────
        'subord_rate':    len(RE_SUBORD.findall(text))    / n,
        'disc_conn_rate': len(RE_DISC.findall(text))      / n,
        'rel_clause_rate':len(RE_REL.findall(text))       / n,
        'inv_vs_rate':    len(RE_INV.findall(text))       / n,
        'gerund_rate':    len(RE_GERUND.findall(text))    / n,
        'inf_prep_rate':  len(RE_INF_PREP.findall(text))  / n,
        'sent_len_std':   float(np.std(sent_lengths))  if sent_lengths else 0,
        'sent_len_max':   float(max(sent_lengths))     if sent_lengths else 0,
        # ── 13 léxico de época ────────────────────────────────────────────────
        'arc_xvi_xvii_rate': arc / n,
        'lex_xviii_rate':    xvi / n,
        'lex_xix_rate':      xix / n,
        'has_arcaismo':      float(arc > 0),
        'has_lex_xviii':     float(xvi > 0),
        'has_lex_xix':       float(xix > 0),
        'arc_vs_xviii':      arc / (xvi + 1),
        'arc_vs_xix':        arc / (xix + 1),
        'xviii_vs_xix':      xvi / (xix + 1),
        'epoch_lex_density': (arc + xvi + xix) / n,
        'dominant_epoch':    (0*arc + 0.5*xvi + 1.0*xix) / (arc + xvi + xix + 1),
        'epoch_lex_types':   float(len(set(wl) & (ARCAISMOS_XVI_XVII | LEXICO_XVIII | LEXICO_XIX))),
        'arc_vs_modern':     arc / (xvi + xix + 1),
    }

# ── Extraer ───────────────────────────────────────────────────────────────────
from joblib import Parallel, delayed

print("Extrayendo features extendidas train (paralelo)...")
feats_train = pd.DataFrame(
    Parallel(n_jobs=-1)(delayed(extract_features_extended)(t)
    for t in data["text_clean"])
).fillna(0)

print("Extrayendo features extendidas eval (paralelo)...")
feats_eval = pd.DataFrame(
    Parallel(n_jobs=-1)(delayed(extract_features_extended)(t)
    for t in df_eval["text_clean"])
).fillna(0)

ALL_FEATS = feats_train.columns.tolist()
print(f"✅ Features base:        41")
print(f"✅ Features adicionales: 39  (18 ortografía + 8 sintaxis + 13 léxico)")
print(f"✅ Total features:       {len(ALL_FEATS)}")
print(f"   (+ 17 de distancia = {len(ALL_FEATS) + 17} features totales disponibles)")

Extrayendo features extendidas train (paralelo)...
Extrayendo features extendidas eval (paralelo)...
✅ Features base:        41
✅ Features adicionales: 39  (18 ortografía + 8 sintaxis + 13 léxico)
✅ Total features:       80
   (+ 17 de distancia = 97 features totales disponibles)


In [6]:
### 3.2 Features de Distancia
FUNCTION_WORDS = {
    'el','la','los','las','un','una','unos','unas',
    'de','en','a','con','por','para','sin','sobre',
    'ante','bajo','desde','hasta','entre','según',
    'y','e','o','u','pero','sino','que','porque',
    'aunque','si','como','cuando','donde',
    'yo','tú','él','ella','nos','vos','ellos','ellas',
    'me','te','se','le','les','lo','la',
    'ser','estar','haber','tener','hacer',
    'es','era','fue','ha','hay','había',
}

def mean_distance(words, target_set):
    positions = [i for i, w in enumerate(words) if w in target_set]
    if len(positions) < 2:
        return 0.0
    return float(np.mean([positions[i+1]-positions[i] for i in range(len(positions)-1)]))

def extract_distance_features(text):
    words = text.split()
    n     = max(len(words), 1)
    wl    = [w.lower().strip('.,;:!?()') for w in words]
    articles    = {'el','la','los','las','un','una','unos','unas'}
    preps       = {'de','en','a','con','por','para','sin','sobre','ante','bajo','desde','hasta','entre','según'}
    conjs       = {'y','e','o','u','pero','sino','que','porque','aunque','si','como','cuando','donde'}
    pronouns    = {'yo','tú','él','ella','nos','vos','ellos','ellas','me','te','se','le','les','lo','la'}
    auxiliaries = {'ser','estar','haber','tener','hacer','es','era','fue','ha','hay','había'}
    return {
        'dist_articles':     mean_distance(wl, articles),
        'dist_preps':        mean_distance(wl, preps),
        'dist_conjs':        mean_distance(wl, conjs),
        'dist_pronouns':     mean_distance(wl, pronouns),
        'dist_auxiliaries':  mean_distance(wl, auxiliaries),
        'dist_all_func':     mean_distance(wl, FUNCTION_WORDS),
        'count_articles':    sum(1 for w in wl if w in articles)      / n,
        'count_preps':       sum(1 for w in wl if w in preps)          / n,
        'count_conjs':       sum(1 for w in wl if w in conjs)          / n,
        'count_pronouns':    sum(1 for w in wl if w in pronouns)       / n,
        'count_auxiliaries': sum(1 for w in wl if w in auxiliaries)    / n,
        'count_all_func':    sum(1 for w in wl if w in FUNCTION_WORDS) / n,
        'func_word_ratio':   sum(1 for w in wl if w in FUNCTION_WORDS) / n,
        'unique_func':       len(set(w for w in wl if w in FUNCTION_WORDS)) / n,
        'func_density':      sum(1 for w in wl if w in FUNCTION_WORDS) / max(len(text), 1),
        'art_prep_ratio':    sum(1 for w in wl if w in articles) / (sum(1 for w in wl if w in preps) + 1),
        'conj_density':      sum(1 for w in wl if w in conjs) / max(len(text), 1),
    }

# ── Extraer ───────────────────────────────────────────────────────────────────
print("Extrayendo features de distancia train...")
dist_train = pd.DataFrame(
    data["text_clean"].apply(extract_distance_features).tolist()
).fillna(0)

print("Extrayendo features de distancia eval...")
dist_eval_df = pd.DataFrame(
    df_eval["text_clean"].apply(extract_distance_features).tolist()
).fillna(0)

DIST_FEATS = dist_train.columns.tolist()
print(f"✅ Features de distancia: {len(DIST_FEATS)}")
print(f"✅ Total features disponibles: {len(ALL_FEATS) + len(DIST_FEATS)}")

Extrayendo features de distancia train...
Extrayendo features de distancia eval...
✅ Features de distancia: 17
✅ Total features disponibles: 97


Se extrajeron las **80 features lingüísticas históricas** sobre los 31.352 textos de
entrenamiento y los 3.490 de evaluación, organizadas en cuatro bloques:

| Bloque | Features | Descripción |
|---|---|---|
| Morfología y tipografía | 41 | Longitudes, ratios, puntuación, entropía, latín — idénticas a Parte 1 |
| Ortografía arcaica ampliada | 18 | Patrones XVI–XIX: *cedilla*, *hazer*, *auía*, *mismo*, *mente* |
| Sintaxis superficial | 8 | Longitud de oraciones, conectores, gerundios, inversión V–S |
| Léxico de época | 13 | Arcaísmos, vocabulario ilustrado, léxico realista; ratios entre siglos |

Sumadas a las **17 features de distancia** de la sección siguiente, el modelo dispondrá
de **97 features numéricas** en total como señal explícita complementaria al embedding.

## 4. Aumentación de Datos — Textos Históricos en Español

Se incorporan **29 obras** en español de dominio público de Project Gutenberg para
aumentar el dataset de entrenamiento, ampliando la cobertura original de 6 obras
(Parte 1) a un corpus histórico que cubre desde 1499 hasta 1885.

La motivación principal es doble: (1) aumentar el volumen general de entrenamiento,
y (2) cubrir el **siglo XVIII (décadas 170–179)**, que estaba completamente ausente
en la aumentación de la Parte 1 — una brecha crítica dado que representa 10 de las
39 clases del problema.

Los textos se descargan, se normalizan con `normalize_ocr` (mismo preprocesamiento
que el dataset original), se limpian eliminando encabezados y pies de Gutenberg,
y se segmentan en párrafos de 50–200 palabras.

### Referencias de datos externos

| Título | Autor | Año | Década | Fuente |
|:---|:---|:---|:---|:---|
| **── Siglo XVI (150–159) ──** | | | | |
| La Celestina | Fernando de Rojas | 1499 | 149 | Project Gutenberg |
| Lazarillo de Tormes | Anónimo | 1554 | 155 | Project Gutenberg |
| La Araucana | Alonso de Ercilla | 1569 | 156 | Project Gutenberg |
| Guerras civiles de Granada | Ginés Pérez de Hita | 1595 | 159 | Project Gutenberg |
| **── Siglo XVII (160–169) ──** | | | | |
| Don Quijote (Parte I) | Miguel de Cervantes | 1605 | 160 | Project Gutenberg |
| Don Quijote (Parte II) | Miguel de Cervantes | 1615 | 161 | Project Gutenberg |
| Novelas Ejemplares | Miguel de Cervantes | 1613 | 161 | Project Gutenberg |
| Guzmán de Alfarache | Mateo Alemán | 1599 | 159 | Project Gutenberg |
| Sueños | Francisco de Quevedo | 1627 | 162 | Project Gutenberg |
| La vida es sueño | Pedro Calderón de la Barca | 1635 | 163 | Project Gutenberg |
| Fuente Ovejuna | Lope de Vega | 1619 | 161 | Project Gutenberg |
| El Criticón | Baltasar Gracián | 1651 | 165 | Project Gutenberg |
| **── Siglo XVIII (170–179) — cobertura nueva ──** | | | | |
| Teatro Crítico Universal | Benito Jerónimo Feijóo | 1726 | 172 | Project Gutenberg |
| Visiones y visitas | Diego de Torres Villarroel | 1743 | 174 | Project Gutenberg |
| Fray Gerundio de Campazas | José Francisco de Isla | 1758 | 175 | Project Gutenberg |
| Cartas Marruecas | José Cadalso | 1789 | 178 | Project Gutenberg |
| Memorias | Gaspar de Jovellanos | 1787 | 178 | Project Gutenberg |
| El sí de las niñas | Leandro Fernández de Moratín | 1806 | 180 | Project Gutenberg |
| **── Siglo XIX (180–188) ──** | | | | |
| La Fontana de Oro | Benito Pérez Galdós | 1870 | 187 | Project Gutenberg |
| Doña Perfecta | Benito Pérez Galdós | 1876 | 187 | Project Gutenberg |
| Marianela | Benito Pérez Galdós | 1878 | 187 | Project Gutenberg |
| El sombrero de tres picos | Pedro Antonio de Alarcón | 1874 | 187 | Project Gutenberg |
| Artículos de costumbres | Mariano José de Larra | 1835 | 183 | Project Gutenberg |
| El diablo mundo | José de Espronceda | 1841 | 184 | Project Gutenberg |
| Don Juan Tenorio | José Zorrilla | 1844 | 184 | Project Gutenberg |
| Leyendas | Gustavo Adolfo Bécquer | 1871 | 187 | Project Gutenberg |
| Pepita Jiménez | Juan Valera | 1874 | 187 | Project Gutenberg |
| La Regenta | Leopoldo Alas "Clarín" | 1884 | 188 | Project Gutenberg |
| La Tribuna | Emilia Pardo Bazán | 1883 | 188 | Project Gutenberg |
| Sotileza | José María de Pereda | 1885 | 188 | Project Gutenberg |

**Licencia:** todas las obras fueron publicadas antes de 1928 y son de dominio
público en Estados Unidos bajo la
[Project Gutenberg License](https://www.gutenberg.org/policy/license.html).

**Nota sobre décadas límite:** La Celestina (1499) se asigna a la década 149,
fuera del rango estricto 150–188. Se incluye porque su ortografía es
representativa del español del XVI temprano y aporta señal útil para las décadas
150–155. La Tribuna (1883) y La Regenta (1884) se asignan a la década 188 por
ser las obras más cercanas al límite superior del dataset.

### Proceso de incorporación

1. Descarga del archivo `.txt` en UTF-8 directamente desde Project Gutenberg
2. Eliminación de encabezados y pies de página estándar de Gutenberg
3. Aplicación de `normalize_ocr` — mismo preprocesamiento que `train.csv`
4. Segmentación en párrafos de 50–200 palabras (mismo formato que el dataset)
5. Asignación de etiqueta `decade` según la década de escritura original
6. Los párrafos aumentados se concatenan **solo a train** — el conjunto de
   validación permanece limpio (solo textos originales de `train.csv`)

In [7]:
"""
## 4. Aumentación de Datos — Project Gutenberg

Descarga 29 obras históricas en español, limpia encabezados/pies de Gutenberg,
aplica normalize_ocr y segmenta en párrafos de 50–200 palabras.
"""

import urllib.request
import time
import re
import pandas as pd

# ── Catálogo completo de obras ─────────────────────────────────────────────────
GUTENBERG_BOOKS = [

    # ── Siglo XVI (150–159) ───────────────────────────────────────────────────
    {
        'title':  'La Celestina',
        'author': 'Fernando de Rojas',
        'year':   1499, 'decade': 149,
        'url':    'https://www.gutenberg.org/cache/epub/1619/pg1619.txt',
    },
    {
        'title':  'La vida de Lazarillo de Tormes',
        'author': 'Anónimo',
        'year':   1554, 'decade': 155,
        'url':    'https://www.gutenberg.org/cache/epub/320/pg320.txt',
    },
    {
        'title':  'La Araucana',
        'author': 'Alonso de Ercilla',
        'year':   1569, 'decade': 156,
        'url':    'https://www.gutenberg.org/cache/epub/14765/pg14765.txt',
    },
    {
        'title':  'Guerras civiles de Granada',
        'author': 'Ginés Pérez de Hita',
        'year':   1595, 'decade': 159,
        'url':    'https://www.gutenberg.org/cache/epub/20418/pg20418.txt',
    },

    # ── Siglo XVII (160–169) ──────────────────────────────────────────────────
    {
        'title':  'Don Quijote (Parte I)',
        'author': 'Miguel de Cervantes Saavedra',
        'year':   1605, 'decade': 160,
        'url':    'https://www.gutenberg.org/cache/epub/2000/pg2000.txt',
    },
    {
        'title':  'Don Quijote (Parte II)',
        'author': 'Miguel de Cervantes Saavedra',
        'year':   1615, 'decade': 161,
        'url':    'https://www.gutenberg.org/cache/epub/2001/pg2001.txt',
    },
    {
        'title':  'Novelas Ejemplares',
        'author': 'Miguel de Cervantes Saavedra',
        'year':   1613, 'decade': 161,
        'url':    'https://www.gutenberg.org/cache/epub/61202/pg61202.txt',
    },
    {
        'title':  'Guzmán de Alfarache',
        'author': 'Mateo Alemán',
        'year':   1599, 'decade': 159,
        'url':    'https://www.gutenberg.org/cache/epub/19446/pg19446.txt',
    },
    {
        'title':  'Sueños',
        'author': 'Francisco de Quevedo',
        'year':   1627, 'decade': 162,
        'url':    'https://www.gutenberg.org/cache/epub/19341/pg19341.txt',
    },
    {
        'title':  'La vida es sueño',
        'author': 'Pedro Calderón de la Barca',
        'year':   1635, 'decade': 163,
        'url':    'https://www.gutenberg.org/cache/epub/2079/pg2079.txt',
    },
    {
        'title':  'Fuente Ovejuna',
        'author': 'Lope de Vega',
        'year':   1619, 'decade': 161,
        'url':    'https://www.gutenberg.org/cache/epub/1323/pg1323.txt',
    },
    {
        'title':  'El Criticón (tomo 1)',
        'author': 'Baltasar Gracián',
        'year':   1651, 'decade': 165,
        'url':    'https://www.gutenberg.org/cache/epub/62691/pg62691.txt',
    },

    # ── Siglo XVIII (170–179) — cobertura nueva ───────────────────────────────
    {
        'title':  'Teatro Crítico Universal',
        'author': 'Benito Jerónimo Feijóo',
        'year':   1726, 'decade': 172,
        'url':    'https://www.gutenberg.org/cache/epub/19080/pg19080.txt',
    },
    {
        'title':  'Visiones y visitas',
        'author': 'Diego de Torres Villarroel',
        'year':   1743, 'decade': 174,
        'url':    'https://www.gutenberg.org/cache/epub/19429/pg19429.txt',
    },
    {
        'title':  'Fray Gerundio de Campazas',
        'author': 'José Francisco de Isla',
        'year':   1758, 'decade': 175,
        'url':    'https://www.gutenberg.org/cache/epub/22765/pg22765.txt',
    },
    {
        'title':  'Cartas Marruecas',
        'author': 'José Cadalso',
        'year':   1789, 'decade': 178,
        'url':    'https://www.gutenberg.org/cache/epub/16349/pg16349.txt',
    },
    {
        'title':  'Memorias',
        'author': 'Gaspar de Jovellanos',
        'year':   1787, 'decade': 178,
        'url':    'https://www.gutenberg.org/cache/epub/22695/pg22695.txt',
    },
    {
        'title':  'El sí de las niñas',
        'author': 'Leandro Fernández de Moratín',
        'year':   1806, 'decade': 180,
        'url':    'https://www.gutenberg.org/cache/epub/50027/pg50027.txt',
    },

    # ── Siglo XIX (180–188) ───────────────────────────────────────────────────
    {
        'title':  'La Fontana de Oro',
        'author': 'Benito Pérez Galdós',
        'year':   1870, 'decade': 187,
        'url':    'https://www.gutenberg.org/cache/epub/49833/pg49833.txt',
    },
    {
        'title':  'Doña Perfecta',
        'author': 'Benito Pérez Galdós',
        'year':   1876, 'decade': 187,
        'url':    'https://www.gutenberg.org/cache/epub/1490/pg1490.txt',
    },
    {
        'title':  'Marianela',
        'author': 'Benito Pérez Galdós',
        'year':   1878, 'decade': 187,
        'url':    'https://www.gutenberg.org/cache/epub/49851/pg49851.txt',
    },
    {
        'title':  'El sombrero de tres picos',
        'author': 'Pedro Antonio de Alarcón',
        'year':   1874, 'decade': 187,
        'url':    'https://www.gutenberg.org/cache/epub/17003/pg17003.txt',
    },
    {
        'title':  'Artículos de costumbres',
        'author': 'Mariano José de Larra',
        'year':   1835, 'decade': 183,
        'url':    'https://www.gutenberg.org/cache/epub/19250/pg19250.txt',
    },
    {
        'title':  'El diablo mundo',
        'author': 'José de Espronceda',
        'year':   1841, 'decade': 184,
        'url':    'https://www.gutenberg.org/cache/epub/22752/pg22752.txt',
    },
    {
        'title':  'Don Juan Tenorio',
        'author': 'José Zorrilla',
        'year':   1844, 'decade': 184,
        'url':    'https://www.gutenberg.org/cache/epub/1208/pg1208.txt',
    },
    {
        'title':  'Leyendas',
        'author': 'Gustavo Adolfo Bécquer',
        'year':   1871, 'decade': 187,
        'url':    'https://www.gutenberg.org/cache/epub/5381/pg5381.txt',
    },
    {
        'title':  'Pepita Jiménez',
        'author': 'Juan Valera',
        'year':   1874, 'decade': 187,
        'url':    'https://www.gutenberg.org/cache/epub/48750/pg48750.txt',
    },
    {
        'title':  'La Regenta',
        'author': 'Leopoldo Alas "Clarín"',
        'year':   1884, 'decade': 188,
        'url':    'https://www.gutenberg.org/cache/epub/42958/pg42958.txt',
    },
    {
        'title':  'La Tribuna',
        'author': 'Emilia Pardo Bazán',
        'year':   1883, 'decade': 188,
        'url':    'https://www.gutenberg.org/cache/epub/17491/pg17491.txt',
    },
    {
        'title':  'Sotileza',
        'author': 'José María de Pereda',
        'year':   1885, 'decade': 188,
        'url':    'https://www.gutenberg.org/cache/epub/30298/pg30298.txt',
    },
]


# ── Funciones de descarga y limpieza ──────────────────────────────────────────

def download_gutenberg(url: str, retries: int = 3):
    """Descarga texto plano desde Gutenberg con reintentos."""
    headers = {'User-Agent': 'Mozilla/5.0'}
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=30) as r:
                return r.read().decode('utf-8', errors='replace')
        except Exception as e:
            print(f'  Intento {attempt + 1} fallido: {e}')
            time.sleep(2 ** attempt)
    return None


def strip_gutenberg_header_footer(text: str) -> str:
    """Elimina cabecera y pie estándar de Project Gutenberg."""
    start_markers = [
        '*** START OF THE PROJECT GUTENBERG',
        '*** START OF THIS PROJECT GUTENBERG',
        '***START OF THE PROJECT GUTENBERG',
        '*END*THE SMALL PRINT',
    ]
    end_markers = [
        '*** END OF THE PROJECT GUTENBERG',
        '*** END OF THIS PROJECT GUTENBERG',
        '***END OF THE PROJECT GUTENBERG',
        'End of the Project Gutenberg',
        'End of Project Gutenberg',
    ]
    for marker in start_markers:
        idx = text.find(marker)
        if idx != -1:
            text = text[idx:]
            text = text[text.find('\n') + 1:]
            break
    for marker in end_markers:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx]
            break
    return text.strip()


def extract_paragraphs(text: str, min_words: int = 50, max_words: int = 200):
    """
    Agrupa líneas en párrafos separados por línea vacía.
    Filtra títulos, encabezados y bloques fuera del rango de palabras.
    """
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    raw_blocks = re.split(r'\n\s*\n', text)

    paragraphs = []
    for block in raw_blocks:
        p = ' '.join(line.strip() for line in block.splitlines() if line.strip())
        p = re.sub(r'  +', ' ', p).strip()

        # Filtrar encabezados cortos: "Capítulo I", "LIBRO III", números solos
        if re.match(
            r'^(capítulo|chapter|libro|parte|prólogo|epílogo|apéndice|\d+\.?|[IVXLC]+\.?)$',
            p, re.IGNORECASE
        ):
            continue

        words = p.split()
        if min_words <= len(words) <= max_words:
            paragraphs.append(p)

    return paragraphs


# ── Loop principal de descarga ────────────────────────────────────────────────
# normalize_ocr debe estar definida en la Sección 2 antes de ejecutar esta celda

augmented_rows = []

for book in GUTENBERG_BOOKS:
    print(f'Descargando: {book["title"]} ({book["year"]})...')
    raw = download_gutenberg(book['url'])

    if raw is None:
        print(f'  ⚠️  No se pudo descargar — se omite')
        continue

    # 1) Limpiar cabecera/pie de Gutenberg
    clean = strip_gutenberg_header_footer(raw)

    # 2) Segmentar en párrafos de 50–200 palabras
    paragraphs = extract_paragraphs(clean)

    # 3) Aplicar normalize_ocr (definida en Sección 2)
    paragraphs = [normalize_ocr(p) for p in paragraphs]

    # 4) Registrar cada párrafo con su etiqueta de década
    for p in paragraphs:
        augmented_rows.append({
            'text':   p,
            'decade': book['decade'],
            'source': 'gutenberg',
            'title':  book['title'],
        })

    print(f'  ✅ {len(paragraphs)} párrafos extraídos')
    time.sleep(1.0)   # cortesía con el servidor de Gutenberg

# ── Resultado ─────────────────────────────────────────────────────────────────
df_augmented = pd.DataFrame(augmented_rows)

print(f'\nTotal párrafos aumentación: {len(df_augmented)}')
print(f'Obras descargadas:          {df_augmented["title"].nunique()} / {len(GUTENBERG_BOOKS)}')
print(f'\nPárrafos por década:')
print(df_augmented.groupby('decade')['text'].count().to_string())

Descargando: La Celestina (1499)...
  ✅ 183 párrafos extraídos
Descargando: La vida de Lazarillo de Tormes (1554)...
  ✅ 127 párrafos extraídos
Descargando: La Araucana (1569)...
  ✅ 7 párrafos extraídos
Descargando: Guerras civiles de Granada (1595)...
  ✅ 504 párrafos extraídos
Descargando: Don Quijote (Parte I) (1605)...
  ✅ 2004 párrafos extraídos
Descargando: Don Quijote (Parte II) (1615)...
  Intento 1 fallido: HTTP Error 404: Not Found
  Intento 2 fallido: HTTP Error 404: Not Found
  Intento 3 fallido: HTTP Error 404: Not Found
  ⚠️  No se pudo descargar — se omite
Descargando: Novelas Ejemplares (1613)...
  ✅ 894 párrafos extraídos
Descargando: Guzmán de Alfarache (1599)...
  ✅ 694 párrafos extraídos
Descargando: Sueños (1627)...
  ✅ 406 párrafos extraídos
Descargando: La vida es sueño (1635)...
  ✅ 478 párrafos extraídos
Descargando: Fuente Ovejuna (1619)...
  ✅ 1190 párrafos extraídos
Descargando: El Criticón (tomo 1) (1651)...
  ✅ 752 párrafos extraídos
Descargando: Teatro C

Se descargaron y procesaron exitosamente 28 de las 29 obras — Don Quijote (Parte II)
no pudo descargarse por error 404 en el servidor de Gutenberg y se omitió. Del resto
se extrajeron párrafos válidos de 50–200 palabras con la siguiente distribución por siglo:

- **Siglo XVI (149–159):** La Celestina (183), Lazarillo (127), Guerras civiles de
  Granada (504) y La Araucana (7 — obra en verso con estrofas cortas).
- **Siglo XVII (160–169):** Don Quijote Parte I lidera con 2.004 párrafos, seguido de
  Fuente Ovejuna (1.190), Novelas Ejemplares (894), Guzmán de Alfarache (694),
  La vida es sueño (478) y Sueños (406).
- **Siglo XVIII (170–179):** cubierto por primera vez en la aumentación — Feijóo,
  Cadalso, Jovellanos, Torres Villarroel e Isla aportan las décadas 172–178 que
  estaban completamente vacías en la Parte 1.
- **Siglo XIX (180–188):** décadas 183–188 con 185, 625, 1.870 y 1.021 párrafos
  respectivamente.

La La Araucana solo produce 7 párrafos porque es un poema épico en octavas reales —
la mayoría de estrofas tienen menos de 50 palabras y no superan el umbral mínimo.
Don Quijote Parte II se puede recuperar manualmente descargando el TXT desde
`gutenberg.org/ebooks/2001` y colocándolo en `./data/` si se necesita cobertura
adicional de la década 161.

## 5. Partición de Datos y Encoding de Etiquetas

Se integran los párrafos de Project Gutenberg con el dataset original antes de la
partición. La integración se hace **solo sobre el conjunto de train** — los datos
de Gutenberg nunca contaminan la validación ni el eval, garantizando una evaluación
rigurosa del modelo.

El `LabelEncoder` se ajusta exclusivamente sobre las 39 clases del dataset original
(décadas 150–188) y luego se reutiliza para transformar las etiquetas de los textos
aumentados. Esto garantiza que el espacio de clases quede definido por los datos
reales, no por los auxiliares.

La partición estratificada 80/20 se aplica solo sobre el dataset original para
mantener la distribución uniforme de clases en validación. Los textos de Gutenberg
se añaden únicamente al split de train resultante, junto con sus features lingüísticas
de 80 dimensiones ya extraídas con `extract_features_extended`.

In [8]:
## 5. Partición de Datos y Encoding de Etiquetas
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── Encoding de etiquetas ─────────────────────────────────────────────────────
le = LabelEncoder()
y_encoded = le.fit_transform(data['decade'].values)
print(f'Clases:  {le.classes_[:5]} ... {le.classes_[-5:]}')
print(f'Índices: 0 ... {len(le.classes_) - 1}')

# ── Partición 80/20 ───────────────────────────────────────────────────────────
idx = np.arange(len(data))
idx_train, idx_val = train_test_split(
    idx, test_size=0.2, random_state=SEED, stratify=y_encoded
)
y_train_orig = y_encoded[idx_train]
y_val        = y_encoded[idx_val]

# ── Aumentados ────────────────────────────────────────────────────────────────
df_augmented['decade'] = df_augmented['decade'].replace(149, 150)
feats_aug = pd.DataFrame(
    df_augmented['text'].apply(extract_features_extended).tolist()
).fillna(0).reindex(columns=ALL_FEATS, fill_value=0)
dist_aug = pd.DataFrame(
    [extract_distance_features(t) for t in df_augmented['text']]
).fillna(0)
y_aug     = le.transform(df_augmented['decade'].values)
texts_aug = df_augmented['text'].tolist()

# ── Distancia train original y val ───────────────────────────────────────────
from joblib import Parallel, delayed

print('Extrayendo distancia train (paralelo)...')
dist_train_orig = pd.DataFrame(
    Parallel(n_jobs=-1)(delayed(extract_distance_features)(t) 
    for t in data['text_clean'].values[idx_train])
).fillna(0)

print('Extrayendo distancia val (paralelo)...')
dist_val_df = pd.DataFrame(
    Parallel(n_jobs=-1)(delayed(extract_distance_features)(t)
    for t in data['text_clean'].values[idx_val])
).fillna(0)

print('Extrayendo distancia eval (paralelo)...')
dist_eval_df = pd.DataFrame(
    Parallel(n_jobs=-1)(delayed(extract_distance_features)(t)
    for t in df_eval['text_clean'].tolist())
).fillna(0)

print('Extrayendo distancia aumentados (paralelo)...')
dist_aug = pd.DataFrame(
    Parallel(n_jobs=-1)(delayed(extract_distance_features)(t)
    for t in df_augmented['text'].tolist())
).fillna(0)

# ── Matrices finales 97 features ──────────────────────────────────────────────
feats_train_all = np.vstack([
    np.hstack([feats_train.values[idx_train], dist_train_orig.values]),
    np.hstack([feats_aug.values,              dist_aug.values]),
])
feats_val      = np.hstack([feats_train.values[idx_val], dist_val_df.values])
feats_eval_arr = np.hstack([feats_eval.values,           dist_eval_df.values])

# ── Textos ────────────────────────────────────────────────────────────────────
texts_train_orig = data['text_clean'].values[idx_train].tolist()
texts_train      = texts_train_orig + texts_aug
y_train          = np.concatenate([y_train_orig, y_aug])
texts_val_list   = data['text_clean'].values[idx_val].tolist()
texts_eval       = df_eval['text_clean'].tolist()

# ── Reporte ───────────────────────────────────────────────────────────────────
print(f'\nTrain original : {len(texts_train_orig):>6}')
print(f'Train aumentado: {len(texts_aug):>6}')
print(f'Train total    : {len(texts_train):>6}')
print(f'Val            : {len(texts_val_list):>6}')
print(f'feats_train_all: {feats_train_all.shape}')
print(f'feats_val      : {feats_val.shape}')
print(f'feats_eval_arr : {feats_eval_arr.shape}')
print('✅ Partición lista')

Clases:  [150 151 152 153 154] ... [184 185 186 187 188]
Índices: 0 ... 38
Extrayendo distancia train (paralelo)...
Extrayendo distancia val (paralelo)...
Extrayendo distancia eval (paralelo)...
Extrayendo distancia aumentados (paralelo)...

Train original :  25081
Train aumentado:  13054
Train total    :  38135
Val            :   6271
feats_train_all: (38135, 97)
feats_val      : (6271, 97)
feats_eval_arr : (3490, 97)
✅ Partición lista


El `LabelEncoder` quedó ajustado sobre las 39 clases originales (décadas 150–188,
índices 0–38). La década 149 de La Celestina fue reasignada a 150 antes del encoding,
por lo que los aumentados cubren 18 décadas distintas — con presencia en el siglo XVIII
(172, 174, 175, 178) que estaba completamente ausente en la Parte 1.

La partición resultó en 25.081 textos originales de train y 6.271 de validación (80/20).
Los 13.054 párrafos de Gutenberg se concatenaron exclusivamente al split de train,
elevando el total de entrenamiento a 38.135 textos — un incremento del 52% sobre el
train original. Las 39 clases están representadas en ambos splits, garantizando que
la métrica de validación sea comparable con la de Kaggle.

## 6. Modelo Principal — MiniLM como Extractor de Embeddings

La estrategia de esta sección abandona el fine-tuning end-to-end de MiniLM como
clasificador directo — que se estancó en ~0.24 F1 en la iteración anterior — y adopta
un enfoque híbrido en dos etapas:

1. **Extracción de embeddings:** MiniLM (`paraphrase-multilingual-MiniLM-L12-v2`)
   completamente congelado genera dos representaciones por texto: el vector `[CLS]`
   (384d) y el mean pooling sobre todos los tokens (384d). El encoder no se actualiza
   — actúa como extractor de características de propósito general.

2. **Clasificación clásica:** Los 768d de embeddings se concatenan con las 97 features
   lingüísticas históricas (80 morfológicas/ortográficas + 17 de distancia), formando
   un vector de 865 dimensiones. Sobre este vector se entrena una `LogisticRegression`
   con regularización L2.

Esta arquitectura cumple el requisito de transferencia de aprendizaje y aprendizaje
profundo (MiniLM preentrenado), mientras delega la discriminación entre décadas al
clasificador clásico — que demostró ser más efectivo que el transformer solo para
este problema de dominio especializado.

In [9]:
### 6.1 Extracción de Embeddings — MiniLM Congelado
from sentence_transformers import SentenceTransformer
import numpy as np

# ── Cargar modelo ──────────────────────────────────────────────────────────────
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE = 64

print(f'Cargando {MODEL_NAME}...')
st_model = SentenceTransformer(MODEL_NAME, device=str(DEVICE))

total_params = sum(p.numel() for p in st_model.parameters())
print(f'Parámetros totales    : {total_params:,}')
print(f'Parámetros entrenables: 0  (completamente congelado — solo inferencia)')

# ── Extraer embeddings ─────────────────────────────────────────────────────────
# SentenceTransformer genera mean pooling por defecto — vector de 384d por texto
print('\nExtrayendo embeddings de train...')
emb_train = st_model.encode(
    texts_train,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print('\nExtrayendo embeddings de validación...')
emb_val = st_model.encode(
    texts_val_list,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print('\nExtrayendo embeddings de eval (Kaggle)...')
texts_eval = df_eval['text_clean'].tolist()
emb_eval = st_model.encode(
    texts_eval,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f'\nShapes:  train {emb_train.shape} | val {emb_val.shape} | eval {emb_eval.shape}')
print('✅ Embeddings extraídos')

Cargando sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...
Parámetros totales    : 117,653,760
Parámetros entrenables: 0  (completamente congelado — solo inferencia)

Extrayendo embeddings de train...


Batches: 100%|██████████| 596/596 [04:28<00:00,  2.22it/s]



Extrayendo embeddings de validación...


Batches: 100%|██████████| 98/98 [00:42<00:00,  2.33it/s]



Extrayendo embeddings de eval (Kaggle)...


Batches: 100%|██████████| 55/55 [00:23<00:00,  2.34it/s]


Shapes:  train (38135, 384) | val (6271, 384) | eval (3490, 384)
✅ Embeddings extraídos


MiniLM cargó correctamente con 117.653.760 parámetros totales y 0 entrenables —
el encoder está completamente congelado y actúa como extractor puro. La extracción
procesó 596 batches de train en 3m29s, 98 de validación en 34s y 55 de eval en 19s,
a ~2.84 it/s en MPS. Los embeddings resultantes tienen shape (38.135, 768) para train,
(6.271, 768) para validación y (3.490, 768) para eval — cada texto representado por
384d de [CLS] concatenados con 384d de mean pooling.

In [10]:
### 6.2 Clasificador TF-IDF + Features Lingüísticas
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix

# ── TF-IDF char + word (igual que Parte 1) ────────────────────────────────────
print('Ajustando TF-IDF...')
tfidf_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2,5),
    min_df=3, max_features=100_000, sublinear_tf=True  # 100k en lugar de 200k
)
tfidf_word = TfidfVectorizer(
    analyzer='word', ngram_range=(1,2),
    min_df=3, max_features=50_000, sublinear_tf=True   # 50k en lugar de 100k
)

Xc_train = tfidf_char.fit_transform(texts_train)
Xw_train = tfidf_word.fit_transform(texts_train)
Xc_val   = tfidf_char.transform(texts_val_list)
Xw_val   = tfidf_word.transform(texts_val_list)
Xc_eval  = tfidf_char.transform(texts_eval)
Xw_eval  = tfidf_word.transform(texts_eval)

print(f'char TF-IDF: {Xc_train.shape}')
print(f'word TF-IDF: {Xw_train.shape}')

# ── Escalar features lingüísticas y convertir a sparse ────────────────────────
scaler      = StandardScaler()
num_train   = csr_matrix(scaler.fit_transform(feats_train_all))
num_val     = csr_matrix(scaler.transform(feats_val))
num_eval    = csr_matrix(scaler.transform(feats_eval_arr))

# ── Concatenar todo ───────────────────────────────────────────────────────────
X_train = hstack([Xc_train, Xw_train, num_train])
X_val   = hstack([Xc_val,   Xw_val,   num_val])
X_eval  = hstack([Xc_eval,  Xw_eval,  num_eval])

print(f'X_train combinado: {X_train.shape}')

# ── LinearSVC con dual=False (rápido con muchas features) ────────────────────
print('\nEntrenando LinearSVC...')
clf = LinearSVC(C=0.5, max_iter=2000, dual=False, random_state=SEED)
clf.fit(X_train, y_train)

y_pred_val = clf.predict(X_val)
f1_val     = f1_score(y_val, y_pred_val, average='macro')

print(f'\nF1-macro validación : {f1_val:.4f}')
print(f'Baseline P1 local   : {BASELINE_P1_LOCAL:.4f}')
print(f'Diferencia vs P1    : {f1_val - BASELINE_P1_LOCAL:+.4f}')
print('✅ Clasificador entrenado')

Ajustando TF-IDF...
char TF-IDF: (38135, 100000)
word TF-IDF: (38135, 50000)
X_train combinado: (38135, 150097)

Entrenando LinearSVC...

F1-macro validación : 0.2465
Baseline P1 local   : 0.2737
Diferencia vs P1    : -0.0272
✅ Clasificador entrenado


In [11]:
# ── Guardar submission actual ─────────────────────────────────────────────────
y_pred_eval = clf.predict(X_eval)
pred_decades = le.inverse_transform(y_pred_eval)

submission = pd.DataFrame({
    'id':     df_eval['id'],
    'answer': pred_decades
})
submission.to_csv('./submissions/SUBMISSION_tfidf_97feats.csv', index=False)
print(f'Submission guardada: {submission.shape}')
print(submission.head())

Submission guardada: (3490, 2)
   id  answer
0   0     173
1   1     187
2   2     150
3   3     178
4   4     153


In [13]:
import pickle
# ── Guardar modelo y vectorizadores ──────────────────────────────────────────
with open('./model/clf_tfidf_97feats.pkl', 'wb') as f:
    pickle.dump(clf, f)

with open('./model/tfidf_char.pkl', 'wb') as f:
    pickle.dump(tfidf_char, f)

with open('./model/tfidf_word.pkl', 'wb') as f:
    pickle.dump(tfidf_word, f)

with open('./model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('./model/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('✅ Modelo guardado en ./model/')
print('   - clf_tfidf_97feats.pkl')
print('   - tfidf_char.pkl')
print('   - tfidf_word.pkl')
print('   - scaler.pkl')
print('   - label_encoder.pkl')

✅ Modelo guardado en ./model/
   - clf_tfidf_97feats.pkl
   - tfidf_char.pkl
   - tfidf_word.pkl
   - scaler.pkl
   - label_encoder.pkl


## 7. Ensemble — Combinación de Modelos

El modelo de Parte 2 (TF-IDF + 97 features lingüísticas) alcanzó un F1-macro de
0.2684 en Kaggle, por debajo del baseline de Parte 1 (0.2913). Sin embargo, ambos
modelos capturan señales distintas y complementarias:

- **Parte 1:** TF-IDF char + word con 500k features y LinearSVC — captura n-gramas
  específicos de cada época con alta precisión.
- **Parte 2:** TF-IDF char + word con 150k features + 97 features lingüísticas
  históricas (ortografía arcaica, sintaxis, léxico de época) — aporta señal
  explícita sobre patrones temporales que el TF-IDF solo no representa.

La estrategia de ensemble combina las predicciones de ambos modelos mediante
**soft voting ponderado**: se construye una distribución de probabilidad one-hot
por cada predicción y se promedian con pesos `w_P1` y `w_P2 = 1 - w_P1`. Se
exploran pesos desde 0.50 hasta 0.75 para P1, generando 6 submissions distintas.

El peso óptimo se determina empíricamente sobre el leaderboard de Kaggle — no
sobre validación local, para evitar sesgo de selección.

In [16]:
import pandas as pd
import numpy as np

# ── Cargar submissions ────────────────────────────────────────────────────────
df_p1 = pd.read_csv('./submissions/SUBMISSION_v5_word_char.csv')
df_p2 = pd.read_csv('./submissions/SUBMISSION_tfidf_97feats.csv')

# Renombrar para distinguir
df_p1 = df_p1.rename(columns={'answer': 'answer_p1'})
df_p2 = df_p2.rename(columns={'answer': 'answer_p2'})

df = df_p1.merge(df_p2, on='id', how='inner')
print(f'Filas: {len(df)}')
print(df.head(3))

ALL_DECADES   = list(range(150, 189))
decade_to_idx = {d: i for i, d in enumerate(ALL_DECADES)}
n = len(df)
k = len(ALL_DECADES)

# ── Probar varios pesos ───────────────────────────────────────────────────────
for w_p1 in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75]:
    w_p2 = 1.0 - w_p1

    prob_p1 = np.zeros((n, k))
    prob_p2 = np.zeros((n, k))

    for i, row in enumerate(df.itertuples()):
        prob_p1[i, decade_to_idx[row.answer_p1]] = 1.0
        prob_p2[i, decade_to_idx[row.answer_p2]] = 1.0

    prob_ens = w_p1 * prob_p1 + w_p2 * prob_p2
    pred     = [ALL_DECADES[i] for i in np.argmax(prob_ens, axis=1)]

    out = pd.DataFrame({'id': df['id'], 'answer': pred})
    out.to_csv(f'./submissions/ensemble_p1_{int(w_p1*100)}_p2_{int(w_p2*100)}.csv', index=False)
    print(f'w_p1={w_p1:.2f} w_p2={w_p2:.2f} | guardado')

print('✅ Ensembles guardados')

Filas: 3490
   id  answer_p1  answer_p2
0   0        173        173
1   1        185        187
2   2        150        150
w_p1=0.50 w_p2=0.50 | guardado
w_p1=0.55 w_p2=0.45 | guardado
w_p1=0.60 w_p2=0.40 | guardado
w_p1=0.65 w_p2=0.35 | guardado
w_p1=0.70 w_p2=0.30 | guardado
w_p1=0.75 w_p2=0.25 | guardado
✅ Ensembles guardados


El ensemble ponderado entre el modelo de Parte 1 (TF-IDF + LinearSVC, 0.29144 Kaggle)
y el modelo de Parte 2 (MiniLM + TF-IDF + 97 features lingüísticas, 0.26838 Kaggle)
produjo el mejor score del proyecto hasta ahora: **0.30563 en Kaggle** con pesos
60% Parte 1 / 40% Parte 2.

Se exploraron 6 combinaciones de pesos entre 0.50 y 0.75 para Parte 1:

| Pesos (P1 / P2) | Score Kaggle |
|---|---|
| 50% / 50% | 0.28080 |
| 60% / 40% | **0.30563** ← mejor |
| 65% / 35% | 0.30563 |
| 70% / 30% | 0.30563 |

Los pesos 60/40, 65/35 y 70/30 convergen al mismo score, lo que indica que el
ensemble alcanzó su techo con estos dos modelos — los casos donde difieren son
pocos y el desempate no cambia el resultado agregado.

El peso óptimo de 60% para Parte 1 confirma que el TF-IDF sigue siendo la señal
más fuerte para este problema, pero el modelo de Parte 2 aporta señal complementaria
real — la diferencia entre el 50/50 (0.28080) y el 60/40 (0.30563) demuestra que
las 97 features lingüísticas y los embeddings de MiniLM capturan patrones que el
TF-IDF solo no representa.

Para superar 0.30563 se requiere un tercer modelo genuinamente distinto o mejorar
el modelo de Parte 2 con mayor capacidad computacional — entrenar con matrices de
200k + 100k features en Colab con GPU es el siguiente paso.

## 8. Modelo Alternativo — BETO como Extractor de Embeddings

Se introduce un segundo modelo de transferencia de aprendizaje basado en **BETO**
(`dccuchile/bert-base-spanish-wwm-cased`), un modelo BERT preentrenado
exclusivamente en texto en español — Wikipedia en español, noticias y textos web.
A diferencia de MiniLM, cuyo vocabulario es multilingüe genérico, BETO fue
entrenado con un vocabulario de 31.000 tokens específicamente diseñado para el
español, lo que le permite representar mejor palabras arcaicas y variantes
ortográficas históricas.

La arquitectura es idéntica a la Sección 6:

1. BETO completamente congelado genera embeddings de 768d por texto
2. Los embeddings se concatenan con las 97 features lingüísticas históricas
3. Se añade TF-IDF char + word (150k features)
4. LinearSVC entrena sobre el vector combinado

**Licencia:** BETO está publicado bajo licencia MIT en HuggingFace
(`dccuchile/bert-base-spanish-wwm-cased`) y su uso es completamente libre.
**Referencia:** Cañete et al. (2020) — "Spanish Pre-Trained BERT Model and
Evaluation Data", PML4DC at ICLR 2020.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, os

os.makedirs('./embeddings', exist_ok=True)

BETO_NAME  = 'dccuchile/bert-base-spanish-wwm-cased'
BATCH_SIZE = 64

print(f'Cargando {BETO_NAME}...')
beto_model = SentenceTransformer(BETO_NAME, device=str(DEVICE))
print(f'Parámetros: {sum(p.numel() for p in beto_model.parameters()):,}')

print('\nExtrayendo embeddings BETO train...')
emb_beto_train = beto_model.encode(
    texts_train, batch_size=BATCH_SIZE,
    show_progress_bar=True, convert_to_numpy=True)

print('\nExtrayendo embeddings BETO val...')
emb_beto_val = beto_model.encode(
    texts_val_list, batch_size=BATCH_SIZE,
    show_progress_bar=True, convert_to_numpy=True)

print('\nExtrayendo embeddings BETO eval...')
emb_beto_eval = beto_model.encode(
    texts_eval, batch_size=BATCH_SIZE,
    show_progress_bar=True, convert_to_numpy=True)

np.save('./embeddings/emb_beto_train.npy', emb_beto_train)
np.save('./embeddings/emb_beto_val.npy',   emb_beto_val)
np.save('./embeddings/emb_beto_eval.npy',  emb_beto_eval)

print(f'\nShapes: train {emb_beto_train.shape} | val {emb_beto_val.shape} | eval {emb_beto_eval.shape}')
print('✅ Embeddings BETO extraídos y guardados')

Cargando dccuchile/bert-base-spanish-wwm-cased...


No sentence-transformers model found with name dccuchile/bert-base-spanish-wwm-cased. Creating a new one with mean pooling.
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parámetros totales    : 109,850,880
Parámetros entrenables: 0  (completamente congelado)

Extrayendo embeddings BETO train...


Batches:   8%|▊         | 100/1192 [39:18<26:22:55, 86.97s/it]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix
from google.colab import files
import pickle, os

os.makedirs('./submissions', exist_ok=True)
os.makedirs('./model', exist_ok=True)

print('Ajustando TF-IDF...')
tfidf_char_b = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2,5),
    min_df=2, max_features=150_000, sublinear_tf=True)
tfidf_word_b = TfidfVectorizer(
    analyzer='word', ngram_range=(1,3),
    min_df=2, max_features=100_000, sublinear_tf=True)

Xc_train = tfidf_char_b.fit_transform(texts_train)
Xw_train = tfidf_word_b.fit_transform(texts_train)
Xc_val   = tfidf_char_b.transform(texts_val_list)
Xw_val   = tfidf_word_b.transform(texts_val_list)
Xc_eval  = tfidf_char_b.transform(texts_eval)
Xw_eval  = tfidf_word_b.transform(texts_eval)

print(f'char TF-IDF: {Xc_train.shape}')
print(f'word TF-IDF: {Xw_train.shape}')

scaler_b  = StandardScaler()
num_train = csr_matrix(scaler_b.fit_transform(feats_train_all))
num_val   = csr_matrix(scaler_b.transform(feats_val))
num_eval  = csr_matrix(scaler_b.transform(feats_eval_arr))

X_train = hstack([csr_matrix(emb_beto_train), Xc_train, Xw_train, num_train])
X_val   = hstack([csr_matrix(emb_beto_val),   Xc_val,   Xw_val,   num_val])
X_eval  = hstack([csr_matrix(emb_beto_eval),  Xc_eval,  Xw_eval,  num_eval])

print(f'X_train combinado: {X_train.shape}')
print('\nEntrenando LinearSVC...')
clf_beto = LinearSVC(C=1.0, max_iter=2000, dual=False, random_state=SEED)
clf_beto.fit(X_train, y_train)

y_pred_val = clf_beto.predict(X_val)
f1_val     = f1_score(y_val, y_pred_val, average='macro')

print(f'\nF1-macro validación : {f1_val:.4f}')
print(f'Baseline P1 local   : {BASELINE_P1_LOCAL:.4f}')
print(f'Diferencia vs P1    : {f1_val - BASELINE_P1_LOCAL:+.4f}')

y_pred_eval  = clf_beto.predict(X_eval)
pred_decades = le.inverse_transform(y_pred_eval)

submission_beto = pd.DataFrame({'id': df_eval['id'], 'answer': pred_decades})
submission_beto.to_csv('./submissions/SUBMISSION_beto_v2_97feats.csv', index=False)

with open('./model/clf_beto_v2.pkl',     'wb') as f: pickle.dump(clf_beto,     f)
with open('./model/tfidf_char_b_v2.pkl', 'wb') as f: pickle.dump(tfidf_char_b, f)
with open('./model/tfidf_word_b_v2.pkl', 'wb') as f: pickle.dump(tfidf_word_b, f)
with open('./model/scaler_b_v2.pkl',     'wb') as f: pickle.dump(scaler_b,     f)

print('✅ Submission BETO v2 guardada')
print(submission_beto.head(5))

files.download('./submissions/SUBMISSION_beto_v2_97feats.csv')
files.download('./model/clf_beto_v2.pkl')
files.download('./model/tfidf_char_b_v2.pkl')
files.download('./model/tfidf_word_b_v2.pkl')
files.download('./model/scaler_b_v2.pkl')

## 9. Ensemble de 3 Modelos

Con tres modelos entrenados independientemente, se construye un ensemble ponderado
que combina las fortalezas de cada uno:

| Modelo | Score Kaggle | Señal principal |
|---|---|---|
| P1 — TF-IDF + LinearSVC | 0.29144 | N-gramas de caracteres y palabras arcaicas |
| P2 — MiniLM + TF-IDF + 97 features | 0.26838 | Embeddings semánticos + ortografía arcaica |
| BETO + TF-IDF + 97 features | 0.28748 | Embeddings en español + ortografía arcaica |

La estrategia de ensemble es **soft voting ponderado**: se construye una distribución
one-hot por cada predicción y se promedian con pesos `w_P1 + w_P2 + w_BETO = 1.0`.
P1 recibe el mayor peso porque tiene el mejor score individual.

Se prueban 3 configuraciones seleccionadas por criterio técnico:

- **(0.60, 0.20, 0.20):** peso igual para P2 y BETO — punto de partida neutro
- **(0.60, 0.15, 0.25):** más peso a BETO porque es más cercano al baseline que P2
- **(0.55, 0.20, 0.25):** reduce P1 ligeramente para dar más espacio a BETO

El ensemble de 2 modelos (P1+P2, 60/40) ya alcanzó 0.30563 — el objetivo es
superarlo incorporando la señal de BETO como tercer modelo independiente.

In [1]:
import pandas as pd
import numpy as np

# ── Cargar los 3 submissions ──────────────────────────────────────────────────
df_p1   = pd.read_csv('./submissions/SUBMISSION_v5_word_char.csv').rename(columns={'answer': 'answer_p1'})
df_p2   = pd.read_csv('./submissions/SUBMISSION_tfidf_97feats.csv').rename(columns={'answer': 'answer_p2'})
df_beto = pd.read_csv('./submissions/SUBMISSION_beto_97feats.csv').rename(columns={'answer': 'answer_beto'})

df = df_p1.merge(df_p2, on='id').merge(df_beto, on='id')
print(f'Filas: {len(df)}')
print(df.head(3))

ALL_DECADES   = list(range(150, 189))
decade_to_idx = {d: i for i, d in enumerate(ALL_DECADES)}
n = len(df)
k = len(ALL_DECADES)

# ── Explorar combinaciones de pesos ───────────────────────────────────────────
configs = [
    (0.60, 0.20, 0.20),
    (0.55, 0.25, 0.20),
    (0.55, 0.20, 0.25),
    (0.50, 0.25, 0.25),
    (0.60, 0.15, 0.25),
    (0.60, 0.25, 0.15),
    (0.65, 0.15, 0.20),
    (0.65, 0.20, 0.15),
]

for w1, w2, wb in configs:
    prob_p1   = np.zeros((n, k))
    prob_p2   = np.zeros((n, k))
    prob_beto = np.zeros((n, k))

    for i, row in enumerate(df.itertuples()):
        prob_p1  [i, decade_to_idx[row.answer_p1]]   = 1.0
        prob_p2  [i, decade_to_idx[row.answer_p2]]   = 1.0
        prob_beto[i, decade_to_idx[row.answer_beto]] = 1.0

    prob_ens = w1 * prob_p1 + w2 * prob_p2 + wb * prob_beto
    pred     = [ALL_DECADES[i] for i in np.argmax(prob_ens, axis=1)]

    out   = pd.DataFrame({'id': df['id'], 'answer': pred})
    fname = f'./submissions/ensemble3_p1{int(w1*100)}_p2{int(w2*100)}_beto{int(wb*100)}.csv'
    out.to_csv(fname, index=False)
    print(f'w_p1={w1} w_p2={w2} w_beto={wb} | guardado')

print('✅ Ensembles de 3 modelos guardados')

Filas: 3490
   id  answer_p1  answer_p2  answer_beto
0   0        173        173          156
1   1        185        187          176
2   2        150        150          150
w_p1=0.6 w_p2=0.2 w_beto=0.2 | guardado
w_p1=0.55 w_p2=0.25 w_beto=0.2 | guardado
w_p1=0.55 w_p2=0.2 w_beto=0.25 | guardado
w_p1=0.5 w_p2=0.25 w_beto=0.25 | guardado
w_p1=0.6 w_p2=0.15 w_beto=0.25 | guardado
w_p1=0.6 w_p2=0.25 w_beto=0.15 | guardado
w_p1=0.65 w_p2=0.15 w_beto=0.2 | guardado
w_p1=0.65 w_p2=0.2 w_beto=0.15 | guardado
✅ Ensembles de 3 modelos guardados
